# VariantForge AI — Dedicated Google Colab T4 GPU Remote Inference Server
### Production GPU Worker for 7 Pretrained AI Models
1. **Set Runtime:** Click `Runtime` -> `Change runtime type` -> select **T4 GPU** -> click `Save`.
2. **Add Ngrok Secret:** On the left sidebar, click the 🔑 (Secrets icon), add secret `NGROK_AUTHTOKEN` with notebook access **ON**.

In [1]:
# STEP 1: Verify Hardware & GPU Acceleration
import torch
!nvidia-smi
print('=' * 60)
print('PyTorch Version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name :', torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Total GPU VRAM  : {total_vram:.2f} GB')
print('=' * 60)

Fri Nov 20 17:51:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# STEP 2: Install Dependencies
!pip install -q --upgrade pip
!pip install -q diffusers transformers accelerate safetensors torchvision
!pip install -q opencv-python pillow numpy fastapi uvicorn pyngrok easyocr paddleocr
!pip install -q bitsandbytes

# Meta SAM 2.1 Repository & Weights
!git clone -q https://github.com/facebookresearch/sam2.git || true
!pip install -q "git+https://github.com/facebookresearch/sam2.git" || true
!mkdir -p checkpoints
!wget -q -nc -O checkpoints/sam2.1_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt || true

print("✅ Dependencies installed successfully!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Dependencies installed successfully!


In [3]:
# STEP 3: Pre-download & Cache All 7 Models
from huggingface_hub import snapshot_download
import os

print("1/7 Downloading SDXL Inpainting 1.0...")
snapshot_download(repo_id="diffusers/stable-diffusion-xl-1.0-inpainting-0.1")

print("2/7 Downloading SDXL ControlNet Canny...")
snapshot_download(repo_id="diffusers/controlnet-canny-sdxl-1.0")

print("3/7 Downloading BRIA RMBG (2.0 / 1.4)...")
try:
    snapshot_download(repo_id="briaai/RMBG-2.0")
except Exception:
    snapshot_download(repo_id="briaai/RMBG-1.4")

print("4/7 Downloading Qwen2.5-VL-3B-Instruct...")
snapshot_download(repo_id="Qwen/Qwen2.5-VL-3B-Instruct")

print("5/7 Downloading FLUX.2 Klein 4B...")
try:
    snapshot_download(repo_id="black-forest-labs/FLUX.2-klein-4B")
except Exception as e:
    print("Note on FLUX gated access:", e)

print("6/7 Meta SAM 2.1 checkpoint: checkpoints/sam2.1_hiera_large.pt ready.")

print("7/7 Initializing PaddleOCR & EasyOCR...")
import easyocr
easyocr.Reader(['en'], gpu=torch.cuda.is_available())

print("=" * 60)
print("🎉 ALL 7 PRETRAINED MODELS DOWNLOADED & CACHED SUCCESSFULLY!")
print("=" * 60)

1/7 Downloading SDXL Inpainting 1.0...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

2/7 Downloading SDXL ControlNet Canny...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

... (10 outputs left out)

Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

6/7 Meta SAM 2.1 checkpoint: checkpoints/sam2.1_hiera_large.pt ready.
7/7 Initializing PaddleOCR & EasyOCR...


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete============================================================
🎉 ALL 7 PRETRAINED MODELS DOWNLOADED & CACHED SUCCESSFULLY!


In [4]:
# STEP 3: Pre-download & Cache All 7 Models
from huggingface_hub import snapshot_download
import os

print("1/7 Downloading SDXL Inpainting 1.0...")
snapshot_download(repo_id="diffusers/stable-diffusion-xl-1.0-inpainting-0.1")

print("2/7 Downloading SDXL ControlNet Canny...")
snapshot_download(repo_id="diffusers/controlnet-canny-sdxl-1.0")

print("3/7 Downloading BRIA RMBG (2.0 / 1.4)...")
try:
    snapshot_download(repo_id="briaai/RMBG-2.0")
except Exception:
    snapshot_download(repo_id="briaai/RMBG-1.4")

print("4/7 Downloading Qwen2.5-VL-3B-Instruct...")
snapshot_download(repo_id="Qwen/Qwen2.5-VL-3B-Instruct")

print("5/7 Downloading FLUX.2 Klein 4B...")
try:
    snapshot_download(repo_id="black-forest-labs/FLUX.2-klein-4B")
except Exception as e:
    print("Note on FLUX gated access:", e)

print("6/7 Meta SAM 2.1 checkpoint: checkpoints/sam2.1_hiera_large.pt ready.")

print("7/7 Initializing PaddleOCR & EasyOCR...")
import easyocr
easyocr.Reader(['en'], gpu=torch.cuda.is_available())

print("=" * 60)
print("🎉 ALL 7 PRETRAINED MODELS DOWNLOADED & CACHED SUCCESSFULLY!")
print("=" * 60)

1/7 Downloading SDXL Inpainting 1.0...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

2/7 Downloading SDXL ControlNet Canny...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

... (8 outputs left out)

5/7 Downloading FLUX.2 Klein 4B...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

6/7 Meta SAM 2.1 checkpoint: checkpoints/sam2.1_hiera_large.pt ready.
7/7 Initializing PaddleOCR & EasyOCR...
🎉 ALL 7 PRETRAINED MODELS DOWNLOADED & CACHED SUCCESSFULLY!


In [10]:
!grep -o 'https://.*\.trycloudflare\.com' tunnel.log | tail -n 1

grep: tunnel.log: No such file or directory


In [ ]:
# Install cloudflared binary
!wget -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version
# ==============================================================================
# VARIANTFORGE AI - PRODUCTION REMOTE T4 GPU WORKER (ZERO-AUTH, NO 403, NO CRASH)
# ==============================================================================
import os, io, gc, sys, base64, time, subprocess, torch
import numpy as np
from PIL import Image
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, List, Dict, Any
import nest_asyncio
import uvicorn

app = FastAPI(title="VariantForge AI Remote Worker", version="3.1.0")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

active_models: Dict[str, Any] = {}

def unload_models_except(keep: str):
    for name in list(active_models.keys()):
        if name != keep:
            del active_models[name]
    cleanup_gpu()

def b64_to_image(b64_str: str) -> Image.Image:
    if "," in b64_str:
        b64_str = b64_str.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str)))

def image_to_b64(img: Image.Image, format: str = "PNG") -> str:
    buf = io.BytesIO()
    img.save(buf, format=format)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

# 1. SDXL Inpainting Loader
def get_sdxl_inpaint():
    if "sdxl_inpaint" not in active_models:
        unload_models_except("sdxl_inpaint")
        from diffusers import AutoPipelineForInpainting
        print("Loading SDXL Inpainting into VRAM...")
        pipe = AutoPipelineForInpainting.from_pretrained(
            "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
            torch_dtype=dtype,
            variant="fp16" if torch.cuda.is_available() else None
        ).to(device)
        if hasattr(pipe, "enable_attention_slicing"):
            pipe.enable_attention_slicing()
        active_models["sdxl_inpaint"] = pipe
    return active_models["sdxl_inpaint"]

# 2. ControlNet SDXL Canny Loader
def get_controlnet():
    if "controlnet" not in active_models:
        unload_models_except("controlnet")
        from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline
        print("Loading ControlNet SDXL Canny into VRAM...")
        c_model = ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=dtype)
        pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            controlnet=c_model,
            torch_dtype=dtype
        ).to(device)
        if hasattr(pipe, "enable_attention_slicing"):
            pipe.enable_attention_slicing()
        active_models["controlnet"] = pipe
    return active_models["controlnet"]

# 3. Salient Object Cutout (RMBG via U2Net - Zero Auth, No 403)
def get_rmbg():
    if "rmbg" not in active_models:
        unload_models_except("rmbg")
        import rembg
        print("Initializing High-Precision Cutout Engine on GPU...")
        active_models["rmbg"] = rembg.new_session("u2net")
    return active_models["rmbg"]

# 4. EasyOCR Loader
ocr_reader = None
def get_ocr():
    global ocr_reader
    if ocr_reader is None:
        import easyocr
        print("Loading EasyOCR on GPU...")
        ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
    return ocr_reader

# ----------------- FastAPI Routes -----------------
@app.get("/health")
def health():
    return {
        "status": "healthy",
        "device": device,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    }

@app.get("/api/models/status")
def models_status():
    tot, alloc = 0, 0
    if torch.cuda.is_available():
        tot = torch.cuda.get_device_properties(0).total_memory / 1e9
        alloc = torch.cuda.memory_allocated(0) / 1e9
    return {
        "connected": True,
        "vram": {
            "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "total_gb": round(tot, 2),
            "allocated_gb": round(alloc, 2),
            "free_gb": round(tot - alloc, 2)
        },
        "models": {
            "sdxl_inpaint": "Available",
            "controlnet": "Available",
            "rmbg": "Available",
            "ocr": "Available"
        }
    }

class ImageOnlyReq(BaseModel):
    image_base64: str

@app.post("/remove-bg")
def remove_bg_endpoint(req: ImageOnlyReq):
    try:
        import rembg
        img = b64_to_image(req.image_base64).convert("RGB")
        session = get_rmbg()
        cutout = rembg.remove(img, session=session)
        return {"status": "success", "image_base64": image_to_b64(cutout)}
    except Exception as e:
        cleanup_gpu()
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/ocr")
def ocr_endpoint(req: ImageOnlyReq):
    try:
        img = b64_to_image(req.image_base64).convert("RGB")
        reader = get_ocr()
        res = reader.readtext(np.array(img))
        extracted = [{"text": t, "confidence": float(c), "box": [[float(p[0]), float(p[1])] for p in b]} for b, t, c in res]
        return {"status": "success", "elements": extracted}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

class InpaintReq(BaseModel):
    image_base64: str
    mask_base64: str
    prompt: str
    negative_prompt: Optional[str] = "blurry, low quality, artifacts"
    num_inference_steps: Optional[int] = 20
    guidance_scale: Optional[float] = 7.5
    strength: Optional[float] = 0.95
    seed: Optional[int] = 42

@app.post("/inpaint")
def inpaint_endpoint(req: InpaintReq):
    try:
        img = b64_to_image(req.image_base64).convert("RGB")
        mask = b64_to_image(req.mask_base64).convert("L")
        w, h = img.size
        w8, h8 = (w // 8) * 8, (h // 8) * 8
        img = img.resize((w8, h8))
        mask = mask.resize((w8, h8))

        pipe = get_sdxl_inpaint()
        gen = torch.Generator(device=device).manual_seed(req.seed) if req.seed else None
        res = pipe(
            prompt=req.prompt,
            negative_prompt=req.negative_prompt,
            image=img,
            mask_image=mask,
            num_inference_steps=req.num_inference_steps,
            guidance_scale=req.guidance_scale,
            strength=req.strength,
            generator=gen
        ).images[0]
        return {"status": "success", "image_base64": image_to_b64(res.resize((w, h)))}
    except Exception as e:
        cleanup_gpu()
        raise HTTPException(status_code=500, detail=str(e))

class ControlNetReq(BaseModel):
    image_base64: str
    canny_base64: str
    prompt: str
    negative_prompt: Optional[str] = "blurry, low quality"
    conditioning_scale: Optional[float] = 0.8
    num_inference_steps: Optional[int] = 20
    guidance_scale: Optional[float] = 7.5
    seed: Optional[int] = 42

@app.post("/controlnet")
def controlnet_endpoint(req: ControlNetReq):
    try:
        canny = b64_to_image(req.canny_base64).convert("RGB")
        w, h = canny.size
        w8, h8 = (w // 8) * 8, (h // 8) * 8
        canny = canny.resize((w8, h8))

        pipe = get_controlnet()
        gen = torch.Generator(device=device).manual_seed(req.seed) if req.seed else None
        res = pipe(
            prompt=req.prompt,
            negative_prompt=req.negative_prompt,
            image=canny,
            controlnet_conditioning_scale=req.conditioning_scale,
            num_inference_steps=req.num_inference_steps,
            guidance_scale=req.guidance_scale,
            generator=gen
        ).images[0]
        return {"status": "success", "image_base64": image_to_b64(res.resize((w, h)))}
    except Exception as e:
        cleanup_gpu()
        raise HTTPException(status_code=500, detail=str(e))

# ----------------- Start Cloudflare Tunnel & Uvicorn -----------------
!pkill -f cloudflared || true
!pkill -f uvicorn || true

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(30):
    line = tunnel_proc.stdout.readline()
    if "trycloudflare.com" in line:
        for part in line.split():
            if "trycloudflare.com" in part:
                public_url = part.strip()
                break
        if public_url:
            break
    time.sleep(0.5)


# Launch cloudflared with output redirected to tunnel.log
!pkill -f cloudflared || true
!cloudflared tunnel --url http://127.0.0.1:8000 > tunnel.log 2>&1 &
import time, re
time.sleep(3)
with open("tunnel.log", "r") as f:
    text = f.read()
urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
public_url = urls[0] if urls else "Check tunnel.log"

print("=" * 75)
print("🔥 VARIANTFORGE AI REMOTE T4 GPU WORKER ONLINE!")
print("👉 COPY THIS PUBLIC URL INTO YOUR LOCAL APP:", public_url)
print("=" * 75)

config = uvicorn.Config(app=app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

File ‘/usr/local/bin/cloudflared’ already there; not retrieving.
cloudflared version 2026.11.1 (built 2026-11-17-15:04 UTC)
^C
^C
^C
🔥 VARIANTFORGE AI REMOTE T4 GPU WORKER ONLINE!
👉 COPY THIS PUBLIC URL INTO YOUR LOCAL APP: https://serum-adam-manuals-therefore.trycloudflare.com


INFO:     Started server process [2054]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     103.217.237.58:0 - "GET /health HTTP/1.1" 200 OK
INFO:     103.217.237.58:0 - "GET /api/models/status HTTP/1.1" 200 OK
Loading EasyOCR on GPU...
INFO:     103.217.237.57:0 - "POST /ocr HTTP/1.1" 200 OK
INFO:     103.217.237.57:0 - "POST /remove-bg HTTP/1.1" 500 Internal Server Error


/tmp/ipykernel_2054/853470867.py:23: RuntimeWarning: coroutine 'Server.serve' was never awaited
  gc.collect()
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading SDXL Inpainting into VRAM...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


... (13 outputs left out)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

INFO:     103.217.237.57:0 - "GET /api/models/status HTTP/1.1" 200 OK
